# 📘 Notebook 03 — SFT Fine-tuning with LoRA

**Goals:**
1. Load base model + CoT training data
2. Apply LoRA adapters (~0.8% trainable params)
3. Train with SFTTrainer for 3 epochs
4. Save LoRA adapter checkpoint

**Runtime:** ~3 hrs on Kaggle T4 | **GPU:** Required (16GB)

## 1. Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import (
    SFT_OUTPUT_DIR, SFT_LORA_SAVE_DIR,
    SFT_EPOCHS, SFT_BATCH_SIZE, SFT_GRAD_ACCUM_STEPS,
    SFT_LEARNING_RATE, SFT_WARMUP_RATIO, SFT_MAX_SEQ_LENGTH,
    COT_DATA_PATH,
)

print(f"Output dir: {SFT_OUTPUT_DIR}")
print(f"LoRA save dir: {SFT_LORA_SAVE_DIR}")
print(f"Epochs: {SFT_EPOCHS}")
print(f"Effective batch size: {SFT_BATCH_SIZE * SFT_GRAD_ACCUM_STEPS}")

## 2. Load Base Model

In [ ]:
from src.model_utils import load_base_model, apply_lora

model, tokenizer = load_base_model()

## 3. Apply LoRA Adapters

In [ ]:
model, lora_config = apply_lora(model)

# Expected: trainable params ~12M / 1.5B total (~0.8%)

## 4. Load CoT Training Data

In [ ]:
from src.data_utils import load_cot_dataset

cot_dataset = load_cot_dataset(COT_DATA_PATH)
print(f"Training samples: {len(cot_dataset)}")

## 5. Configure & Run SFT Training

> **⏱️ This takes ~3 hours on a Kaggle T4 GPU.**

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM_STEPS,
    learning_rate=SFT_LEARNING_RATE,
    warmup_ratio=SFT_WARMUP_RATIO,
    lr_scheduler_type="cosine",
    bf16=False,
    fp16=True,               # T4/P100 don't support bf16
    logging_steps=10,
    save_steps=100,
    max_seq_length=SFT_MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=cot_dataset,
    args=training_args,
    peft_config=lora_config,
)

print("🚀 Starting SFT training...")
trainer.train()

## 6. Save LoRA Adapter

In [ ]:
trainer.save_model(SFT_LORA_SAVE_DIR)
print(f"✅ SFT training complete. LoRA adapter saved to: {SFT_LORA_SAVE_DIR}")

## 7. Quick Validation — Test the Fine-tuned Model

In [ ]:
from src.model_utils import generate_judgment

sample_facts = """The accused was charged under Section 302 IPC for
allegedly murdering his neighbor during a property dispute.
The prosecution presented three eyewitnesses and forensic evidence.
The defense argued it was a case of self-defense under Section 96 IPC."""

print("=== FINE-TUNED MODEL OUTPUT ===")
output = generate_judgment(model, tokenizer, sample_facts)
print(output)

## 8. Summary

✅ **Completed:**
- Base model loaded with 4-bit quantization
- LoRA adapters applied (~0.8% trainable params)
- SFT training completed on CoT data (3 epochs)
- LoRA checkpoint saved

**Next → Notebook 04:** GRPO reinforcement learning fine-tuning